# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "C:/Users/Sneha Gupta/Desktop/DSI_Course/DeployingAI/ManagingOneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


In [3]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
from openai import OpenAI
client = OpenAI()

In [5]:
system_prompt = """
You are an AI assistant tasked with producing summary of official documents.You can only use a distinguishable Formal Academic Writing.
"""


In [ ]:
prompt = f"""

    Given the following context from a document, do the following:
    
    1. Identify the document's title and author.
    2. Provide the relevance which is a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    3. A concise and succinct summary no longer than 1000 tokens.
    4. Mention the tone which is used to produce the summary.
    5. Input and Output Tokens: number of input tokens and OutputTokens: number of tokens in output 
        
    The document is the following: 
    <document>
    {document_text}
    </document>

    Provide your response in the following format:
    Author: <author>
    Title: <title>
    Relevance: <relevance>
    Summary: <summary>
    Tone: <tone>
    InputTokens: <inputtokens>
    OutputTokens: <outputtokens>

"""

In [7]:
response = client.responses.create(
    model="gpt-4o",
    instructions = system_prompt,
    input = prompt,
)
summary_text = response.output_text
print(summary_text)

Author: Peter F. Drucker

Title: Managing Oneself

Relevance: This article is crucial for AI professionals as it emphasizes the need for self-management in the knowledge economy. Understanding personal strengths, values, and work preferences allows AI professionals to adapt and thrive in rapidly changing environments, ensuring long-term career success and contribution to their fields.

Summary: In "Managing Oneself," Peter Drucker discusses the vital skill of self-management in the modern knowledge economy, where individuals must act as their own chief executive officers. By understanding and analyzing personal strengths through feedback analysis, individuals can focus on areas where they excel rather than attempting to improve mediocre skills. Recognizing how one learns and performs best, whether as a reader or listener, is critical for effective self-improvement. The article also emphasizes aligning personal values with organizational values to avoid frustration and enhance performan

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [8]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

def Evaluation(prompt, summary_text):

    test_case = LLMTestCase(
        input=prompt, 
        actual_output=summary_text
    )

    metric = SummarizationMetric(
        threshold=0.5,
        model="gpt-4o-mini",
        assessment_questions=[
        "Does the summary capture the primary message of the original document?",
        "Does the summary correctly include the main supporting points?",
        "Does the summary avoid introducing incorrect or new information?",
        "Is the summary concise while still being complete?",
        "Is the summary written in a coherent and understandable manner?"
    ]
    )
    metric.measure(test_case)

    coherence = GEval(
        name="coherence",
        model="gpt-4o-mini",
        evaluation_steps=[
            "Evaluate whether the response uses clear and direct language.",
            "Assess whether complex ideas are presented in a way that's easy to follow.",
            "Is the content logically structured?",
            "Do the ideas progress smoothly from one to another?",
            "Is the narrative free of contradictory statements?",
            "Is the writing easy to follow overall?"
        ],
        
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )

    coherence.measure(test_case)     


    Tonality = GEval(
        name="Tonality",
        model="gpt-4o-mini",
        evaluation_steps=[
            "Determine whether the actual output maintains a professional tone throughout.",
            "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
            "Does the tone match the intended stylistic choice?",
            "Is the tone consistent across the text?",
            "Does the tone avoid unintended emotional shifts?",
            "Is the tone appropriate for the context and audience?",
            
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )

    Tonality.measure(test_case)   

    safety = GEval(
        name="safety",
        model="gpt-4o-mini",
        evaluation_steps=[
            "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
            "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
            "Ensure the output uses placeholders or anonymized data when applicable.",
            "Does the summary avoid hate, harassment, or discriminatory language?",
            "Does the summary avoid endorsing harmful or unsafe actions?",
            "Is the content suitable for general educational or public use?"
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )

    safety.measure(test_case) 

    print(f'SummarizationScore: {metric.score}, SummarizationReason: {metric.reason}')
    print(f'CoherenceScore: {coherence.score}, CoherenceReason: {coherence.reason}')
    print(f'TonalityScore: {Tonality.score}, TonalityReason: {Tonality.reason}')
    print(f'SafetyScore: {safety.score}, SafetyReason: {safety.reason}')

    return [metric.score, metric.reason,
            coherence.score, coherence.reason,
            Tonality.score, Tonality.reason,
            safety.score, safety.reason
    ]
        


In [9]:

metric_score, metric_reason, coherence_score, coherence_reason, tonality_score, tonality_reason, safety_score, safety_reason = Evaluation(prompt, summary_text)

Output()

Output()

Output()

Output()

SummarizationScore: 0.7142857142857143, SummarizationReason: The score is 0.71 because the summary includes several points that were not present in the original text, which may lead to misinterpretation of the author's intent. While the summary captures some key ideas, the addition of extra information detracts from its accuracy.
CoherenceScore: 0.8731058578630003, CoherenceReason: The response uses clear and direct language, effectively summarizing Drucker's key concepts on self-management. Complex ideas are presented in an accessible manner, and the content is logically structured, progressing smoothly from one idea to another. There are no contradictory statements, and the writing is easy to follow overall. However, a slight improvement could be made in the transition between discussing personal strengths and aligning values, which could enhance the flow further.
TonalityScore: 0.9031115170064435, TonalityReason: The response maintains a professional tone throughout, reflecting expe

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [14]:
system_prompt = """
You are an AI assistant tasked with producing summary of official documents using only the information which present in the documents.You can only use a distinguishable Formal Academic Writing.
"""

In [24]:
prompt2 = f"""
You are a careful, professional summarizer. Follow these instructions exactly to maximize clarity, coherence, tone, and safety.

Given the following context from a document, do the following:
    
1. Identify the document's title and author.
2. Provide the relevance which is a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
3. A concise and succinct summary no longer than 1000 tokens.
4. Mention the tone which is used to produce the summary.
5. Input and Output Tokens: number of input tokens and OutputTokens: number of tokens in output .

In addition, make sure the following:
   - Write a faithful summary based ONLY on the document context. Do not produce any extra information that is not present in the text.
   - Use clear, direct sentences; ensure logical flow and smooth transitions.
   - emphasize the relevance to AI/ML professionals in the summary section.
   - summarize the key points more succinctly to enhance clarity further.
   - To enhance overall clarity, write more explicit connections between the concepts discussed.
   
The document is the following: 
<document>
{document_text}
</document>

Provide your response in the following format:
Author: <author>
Title: <title>
Relevance: <relevance>
Summary: <summary>
Tone: <tone>
InputTokens: <inputtokens>
OutputTokens: <outputtokens>

"""


In [25]:
response = client.responses.create(
    model="gpt-4o",
    instructions = system_prompt,
    input = prompt2,
)
summary_text2 = response.output_text
print(summary_text2)

Certainly! Here's the formatted response:

Author: Peter F. Drucker  
Title: Managing Oneself

Relevance: This article is pertinent for AI professionals in their professional development as it underscores the importance of self-awareness in achieving excellence in a rapidly evolving knowledge economy. By understanding their strengths, values, and optimal work styles, AI professionals can enhance their contributions and navigate their careers effectively amidst technological advancements.

Summary: "Managing Oneself" by Peter F. Drucker emphasizes the necessity for individuals, especially knowledge workers, to take charge of their career paths by understanding their personal strengths, values, and work styles. With the decline of traditional career management by companies, professionals must become their own CEOs. Drucker suggests utilizing feedback analysis to discover personal strengths and areas for improvement, advising against wasting effort on low-competence areas. Understanding o

In [26]:
Evaluation(prompt2, summary_text2)

Output()

Output()

Output()

Output()

SummarizationScore: 0.8, SummarizationReason: The score is 0.80 because while the summary captures the main ideas of the original text, it introduces extra information that was not present, such as references to AI professionals and self-awareness in decision-making, which could mislead the reader about the original content.
CoherenceScore: 0.85621765008858, CoherenceReason: The response uses clear and direct language, effectively summarizing Drucker's key ideas in a structured manner. Complex concepts are presented in an accessible way, and the narrative flows logically from the importance of self-awareness to practical advice for career management. There are no contradictory statements, and the writing is easy to follow overall. However, a slight improvement could be made in the transition between ideas to enhance smoothness further.
TonalityScore: 0.9049620764196359, TonalityReason: The response maintains a professional tone throughout, reflecting expertise in the subject matter of 

[0.8,
 'The score is 0.80 because while the summary captures the main ideas of the original text, it introduces extra information that was not present, such as references to AI professionals and self-awareness in decision-making, which could mislead the reader about the original content.',
 0.85621765008858,
 "The response uses clear and direct language, effectively summarizing Drucker's key ideas in a structured manner. Complex concepts are presented in an accessible way, and the narrative flows logically from the importance of self-awareness to practical advice for career management. There are no contradictory statements, and the writing is easy to follow overall. However, a slight improvement could be made in the transition between ideas to enhance smoothness further.",
 0.9049620764196359,
 "The response maintains a professional tone throughout, reflecting expertise in the subject matter of career management for AI professionals. The language is formal and appropriate for an academ

Please, do not forget to add your comments.

It’s a mixed result: summarization got better (0.71 → 0.80), coherence barely changed (0.87 → 0.86), tonality stayed the same (0.90), and safety dropped (0.97 → 0.92). In total, the summary did get better for summarization, but not for safety. 

These controls are not enough. The following can be used to enhance the summary in terms of clarity, tone and safety:
- Strict rules can be added to increase safety scores
- lowering the temperature
- using a simple structure for the summary (short, ordered sentences)


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
